In [ ]:
!pip install -q kagglehub
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms
import os

# Check if GPU is active
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

if device.type == 'cpu':
    print("⚠️ WARNING: You are running on a CPU! Go to Runtime > Change runtime type > T4 GPU.")
else:
    print("🚀 GPU is successfully enabled and ready!")

Using device: cuda
🚀 GPU is successfully enabled and ready!


In [ ]:
import kagglehub

# Automatically download the dataset via Kaggle Hub
path = kagglehub.dataset_download("pacificrm/car-insurance-fraud-detection")
print("Dataset downloaded to path:", path)

# Automatically find train and test directories
train_dir, test_dir = None, None
for root, dirs, files in os.walk(path):
    if 'train' in dirs and 'test' in dirs:
        train_dir = os.path.join(root, 'train')
        test_dir = os.path.join(root, 'test')
        break

if not train_dir:
    train_dir = os.path.join(path, 'Insurance-Fraud-Detection', 'Insurance-Fraud-Detection', 'train')
    test_dir = os.path.join(path, 'Insurance-Fraud-Detection', 'Insurance-Fraud-Detection', 'test')

print(f"Train directory: {train_dir}")
print(f"Test directory: {test_dir}")

100%|██████████| 826M/826M [00:06<00:00, 125MB/s] 

Extracting files...


Dataset downloaded to path: /root/.cache/kagglehub/datasets/pacificrm/car-insurance-fraud-detection/versions/2
Train directory: /root/.cache/kagglehub/datasets/pacificrm/car-insurance-fraud-detection/versions/2/Insurance-Fraud-Detection/Insurance-Fraud-Detection/train
Test directory: /root/.cache/kagglehub/datasets/pacificrm/car-insurance-fraud-detection/versions/2/Insurance-Fraud-Detection/Insurance-Fraud-Detection/test


In [ ]:
# ResNet50 requires 224x224 input size and specific ImageNet normalization
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

train_dataset = datasets.ImageFolder(train_dir, data_transforms['train'])
val_dataset = datasets.ImageFolder(test_dir, data_transforms['val'])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

class_names = train_dataset.classes
print(f"Detected Classes: {class_names}")

Detected Classes: ['Fraud', 'Non-Fraud']


In [ ]:
# Load pre-trained ResNet50 weights
weights = models.ResNet50_Weights.DEFAULT
model = models.resnet50(weights=weights)

# Freeze base feature extractor layers
for param in model.parameters():
    param.requires_grad = False

# Replace the final classification head for your classes
num_classes = len(class_names)
model.fc = nn.Sequential(
    nn.Linear(model.fc.in_features, 256),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(256, num_classes)
)

model = model.to(device)
print("ResNet50 successfully loaded and moved to GPU!")

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 176MB/s]


ResNet50 successfully loaded and moved to GPU!


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

num_epochs = 5

for epoch in range(num_epochs):
    print(f'\nEpoch {epoch+1}/{num_epochs}')
    print('-' * 25)

    for phase in ['train', 'val']:
        if phase == 'train':
            model.train()
            dataloader = train_loader
        else:
            model.eval()
            dataloader = val_loader

        running_loss = 0.0
        running_corrects = 0

        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            with torch.set_grad_enabled(phase == 'train'):
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)

                if phase == 'train':
                    loss.backward()
                    optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)

        epoch_loss = running_loss / len(dataloader.dataset)
        epoch_acc = running_corrects.double() / len(dataloader.dataset)

        print(f'{phase.capitalize()} Loss: {epoch_loss:.4f} | Acc: {epoch_acc:.4f}')

print("\nTraining complete!")


Epoch 1/5
-------------------------
Train Loss: 0.1497 | Acc: 0.9592
Val Loss: 0.1997 | Acc: 0.9343

Epoch 2/5
-------------------------
Train Loss: 0.1080 | Acc: 0.9617
Val Loss: 0.1668 | Acc: 0.9350

Epoch 3/5
-------------------------
Train Loss: 0.0992 | Acc: 0.9658
Val Loss: 0.1892 | Acc: 0.9400

Epoch 4/5
-------------------------
Train Loss: 0.0892 | Acc: 0.9675
Val Loss: 0.1493 | Acc: 0.9421

Epoch 5/5
-------------------------
Train Loss: 0.0849 | Acc: 0.9710
Val Loss: 0.1594 | Acc: 0.9449

Training complete!


In [ ]:
from PIL import Image
import torch
import torch.nn.functional as F

# 1. Path to your test image
# (Upload an image using Colab's file explorer on the left, then copy its path)
image_path = '/test1.jpg'  # Replace with your actual image path

# 2. Define the same preprocessing transforms used for validation
inference_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# 3. Load and transform the image
try:
    image = Image.open(image_path).convert('RGB')
    input_tensor = inference_transform(image)
    input_batch = input_tensor.unsqueeze(0)  # Add batch dimension (1, 3, 224, 224)
    input_batch = input_batch.to(device)

    # 4. Run inference
    model.eval()
    with torch.no_grad():
        outputs = model(input_batch)
        probabilities = F.softmax(outputs[0], dim=0)

        # Get highest probability class and its confidence score
        confidence, predicted_idx = torch.max(probabilities, dim=0)
        predicted_class = class_names[predicted_idx.item()]

    # 5. Display results
    print(f"🎯 Predicted Class: {predicted_class}")
    print(f"📊 Confidence: {confidence.item() * 100:.2f}%")

    # Optional: Display all class probabilities
    print("\n--- All Class Probabilities ---")
    for i, name in enumerate(class_names):
        print(f"{name}: {probabilities[i].item() * 100:.2f}%")

except FileNotFoundError:
    print(f"❌ Error: Could not find image at '{image_path}'. Please check the path and try again.")

🎯 Predicted Class: Non-Fraud
📊 Confidence: 84.64%

--- All Class Probabilities ---
Fraud: 15.36%
Non-Fraud: 84.64%


In [ ]:
model_save_path = 'car_fraud_resnet50.pth'

# Save model weights and class mapping
torch.save({
    'model_state_dict': model.state_dict(),
    'class_names': class_names
}, model_save_path)

print(f"Model successfully saved to '{model_save_path}'!")

# Uncomment the line below if you want to download the trained file straight to your computer:
from google.colab import files
files.download(model_save_path)

Model successfully saved to 'car_fraud_resnet50.pth'!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>